# Feature Manipulation

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
import seaborn as sns

#sns.set_theme()

testbjdhjhjhaj

In [ ]:
data = pd.read_parquet(Path("data") / "train.parquet")
data.head()

In [ ]:
data.info()

In [ ]:
def _encode_dates(X):
    X = X.copy()  # Ensure we're working on a copy
    # Encode the date information
    X["year"] = X["date"].dt.year
    X["month"] = X["date"].dt.month
    X["day"] = X["date"].dt.day
    X["weekday"] = X["date"].dt.weekday  # 0=Monday, 6=Sunday
    X["hour"] = X["date"].dt.hour
    # Keep the rest of the columns as they are
    return X

# Apply the encoding function to the dataset
data = data.copy()  # Ensure we're working on a copy
data = _encode_dates(data)

data["weekend"] = (data["weekday"] > 4).astype(int)  # 1 stands for weekend, 0 stands for no weekend
data.head()


In [ ]:
import holidays

# Define French holidays
FR_holidays = holidays.FR(years=range(2019, 2022))

data["FR_holidays"] = data["date"].dt.date.isin(FR_holidays).astype(int)
print(f"Number of rows marked as holidays: {data['FR_holidays'].sum()}")

# Adding Weather Data

In [ ]:
weather_data = pd.read_csv(Path("data") / "external_data.csv")

weather_data["date"] = pd.to_datetime(weather_data["date"], errors="coerce")
print(weather_data["date"].isna().sum())
weather_data = _encode_dates(weather_data)

# Drop the duplicate rows based on the 'date' column
weather_data = weather_data.drop_duplicates(subset="date")

# Verify that the duplicate is removed
duplicate_rows = weather_data[weather_data["date"].duplicated(keep=False)]
print(f"Number of duplicate rows after dropping: {len(duplicate_rows)}")

# Interpolate linearly to get from 3 hour data to 1 hour data
weather_data.set_index("date", inplace=True)  # Set date as the index
weather_data = weather_data.resample("H").interpolate(method="linear")  # Interpolate missing values
weather_data.reset_index(inplace=True)  # Reset index

# Check the new shape of the weather data
print(f"Resampled Weather Data Shape: {weather_data.shape}")

# Merge bike data with weather data using a left join
merged_data = pd.merge(data, weather_data, on="date", how="left")

# Drop redundant date columns to avoid duplicates in the final dataset
merged_data = merged_data.loc[:, ~merged_data.columns.str.endswith(("_x", "_y"))]  # Change 1: Drop `_x` or `_y` suffix columns

# Check the merged dataset
print(f"Merged Data Shape: {merged_data.shape}")


In [ ]:
from sklearn.preprocessing import FunctionTransformer

date_encoder = FunctionTransformer(_encode_dates, validate=False)
sample_encoded = date_encoder.fit_transform(merged_data[["date"]]).head()
sample_encoded

In [ ]:
# Reapply the _encode_dates function to extract date-related columns
merged_data = _encode_dates(merged_data)

# Verify the new columns
print(merged_data[["date", "year", "month", "day", "weekday", "weekend"]].head())


Total changes in features extraction:
1. Encode the date information to single columns (year, month, day, weekday, hour)
2. Add a column indicating weekends (1 = weekend, 0 = no weekend)
3. Add a column of French holidays (1 = French holidays, 0 = no French holidays)
4. Encode a "date" column to match scikit learn requirements.

In [ ]:
# Save the processed_data file in the data folder
merged_data.to_parquet(Path("data") / "train_processed.parquet")